In [1]:
!pip install catboost lightgbm xgboost category_encoders -q


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import KFold
from sklearn.metrics import r2_score

from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor

import warnings
warnings.filterwarnings('ignore')

In [4]:
train = pd.read_csv("Dataset/train.csv")
test = pd.read_csv("Dataset/test.csv")

print(train.shape)
print(test.shape)

train.head()

(77299, 11)
(41778, 10)


,Index,geohash,day,timestamp,demand,RoadType,NumberofLanes,LargeVehicles,Landmarks,Temperature,Weather
0,0,qp02z1,48,0:0,0.048804,NaN,1,Not Allowed,No,NaN,NaN
1,1,qp02zt,48,0:0,0.118507,Residential,3,Allowed,Yes,31.104565,Sunny
2,2,qp08bj,48,0:0,0.027132,Residential,1,Not Allowed,No,25.919267,Sunny
3,3,qp08gt,48,0:0,0.003272,Residential,1,Not Allowed,No,NaN,Rainy
4,4,qp02zq,48,0:0,0.010819,Residential,1,Not Allowed,No,10.803667,Rainy


In [5]:
print(train.isnull().sum())
print(test.isnull().sum())

Index               0
geohash             0
day                 0
timestamp           0
demand              0
RoadType          600
NumberofLanes       0
LargeVehicles       0
Landmarks           0
Temperature      2495
Weather           797
dtype: int64
Index               0
geohash             0
day                 0
timestamp           0
RoadType          324
NumberofLanes       0
LargeVehicles       0
Landmarks           0
Temperature      1349
Weather           431
dtype: int64


In [6]:
for col in ['RoadType','Weather']:

    train[col] = train[col].fillna('Unknown')
    test[col] = test[col].fillna('Unknown')

temp_median = train['Temperature'].median()

train['Temperature'] = train['Temperature'].fillna(temp_median)
test['Temperature'] = test['Temperature'].fillna(temp_median)

In [7]:
def create_features(df):

    temp = df.copy()

    temp[['hour','minute']] = temp['timestamp'].str.split(
        ':',
        expand=True
    ).astype(int)

    temp['is_peak'] = (
        ((temp['hour'] >= 7) & (temp['hour'] <= 10))
        |
        ((temp['hour'] >= 16) & (temp['hour'] <= 20))
    ).astype(int)

    temp['hour_sin'] = np.sin(
        2*np.pi*temp['hour']/24
    )

    temp['hour_cos'] = np.cos(
        2*np.pi*temp['hour']/24
    )

    temp['minute_sin'] = np.sin(
        2*np.pi*temp['minute']/60
    )

    temp['minute_cos'] = np.cos(
        2*np.pi*temp['minute']/60
    )

    return temp

In [8]:
train = create_features(train)
test = create_features(test)

In [9]:
freq = train['geohash'].value_counts()

train['geo_freq'] = train['geohash'].map(freq)
test['geo_freq'] = test['geohash'].map(freq).fillna(0)

In [10]:
cat_cols = [
    'geohash',
    'RoadType',
    'Weather',
    'LargeVehicles',
    'Landmarks'
]

for col in cat_cols:

    combined = pd.concat([
        train[col].astype(str),
        test[col].astype(str)
    ])

    mapping = {
        k:v
        for v,k in enumerate(
            combined.unique()
        )
    }

    train[col] = train[col].astype(str).map(mapping)
    test[col] = test[col].astype(str).map(mapping)

In [11]:
TARGET = "demand"

DROP = [
    "Index",
    "timestamp"
]

X = train.drop(
    columns=DROP+[TARGET]
)

y = train[TARGET]

X_test = test.drop(
    columns=DROP
)

print(X.shape)
print(X_test.shape)

(77299, 16)
(41778, 16)


In [12]:
kf = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

oof_cat = np.zeros(len(X))
pred_cat = np.zeros(len(X_test))

scores = []

In [13]:
for fold,(trn_idx,val_idx) in enumerate(
    kf.split(X)
):

    X_train = X.iloc[trn_idx]
    X_valid = X.iloc[val_idx]

    y_train = y.iloc[trn_idx]
    y_valid = y.iloc[val_idx]

    model = CatBoostRegressor(
        iterations=4000,
        learning_rate=0.03,
        depth=10,
        loss_function='RMSE',
        verbose=0
    )

    model.fit(
        X_train,
        y_train
    )

    val_pred = model.predict(X_valid)

    oof_cat[val_idx] = val_pred

    pred_cat += (
        model.predict(X_test)
        / kf.n_splits
    )

    score = r2_score(
        y_valid,
        val_pred
    )

    scores.append(score)

    print(
        f"Fold {fold+1} R2:",
        score
    )

Fold 1 R2: 0.9268276918087035
Fold 2 R2: 0.92050604570509
Fold 3 R2: 0.9226679960258276
Fold 4 R2: 0.9238237557811585
Fold 5 R2: 0.9280576733755135


In [14]:
print(
    "CatBoost CV:",
    np.mean(scores)
)

CatBoost CV: 0.9243766325392586


In [15]:
oof_lgb = np.zeros(len(X))
pred_lgb = np.zeros(len(X_test))

scores = []

for fold,(trn_idx,val_idx) in enumerate(
    kf.split(X)
):

    model = LGBMRegressor(
        n_estimators=3000,
        learning_rate=0.03,
        max_depth=10,
        num_leaves=64,
        random_state=42
    )

    model.fit(
        X.iloc[trn_idx],
        y.iloc[trn_idx]
    )

    val_pred = model.predict(
        X.iloc[val_idx]
    )

    oof_lgb[val_idx] = val_pred

    pred_lgb += (
        model.predict(X_test)
        / kf.n_splits
    )

    score = r2_score(
        y.iloc[val_idx],
        val_pred
    )

    scores.append(score)

print(
    "LightGBM CV:",
    np.mean(scores)
)

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.005406 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 717
[LightGBM] [Info] Number of data points in the train set: 61839, number of used features: 16
[LightGBM] [Info] Start training from score 0.093784
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: 

In [16]:

oof_xgb = np.zeros(len(X))
pred_xgb = np.zeros(len(X_test))

scores = []

for fold,(trn_idx,val_idx) in enumerate(
    kf.split(X)
):

    model = XGBRegressor(
        n_estimators=3000,
        learning_rate=0.03,
        max_depth=10,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42
    )

    model.fit(
        X.iloc[trn_idx],
        y.iloc[trn_idx]
    )

    val_pred = model.predict(
        X.iloc[val_idx]
    )

    oof_xgb[val_idx] = val_pred

    pred_xgb += (
        model.predict(X_test)
        / kf.n_splits
    )

    score = r2_score(
        y.iloc[val_idx],
        val_pred
    )

    scores.append(score)

print(
    "XGBoost CV:",
    np.mean(scores)
)

XGBoost CV: 0.9251807812013153


In [17]:
final_predictions = (
    pred_cat * 0.5 +
    pred_lgb * 0.25 +
    pred_xgb * 0.25
)

In [21]:
print(type(test))
print(test.shape)

<class 'pandas.DataFrame'>
(41778, 18)


In [22]:
print(type(final_predictions))
print(len(final_predictions))

<class 'numpy.ndarray'>
41778


In [23]:
submission = pd.DataFrame({
    "Index": test["Index"],
    "demand": final_predictions
})

In [24]:
submission.head()

,Index,demand
0,0,0.052345
1,1,0.025225
2,2,0.003106
3,3,0.023834
4,4,0.049234


In [25]:
submission.to_csv("submission.csv", index=False)
print("Saved Successfully")

Saved Successfully


In [27]:
import pandas as pd

sub = pd.read_csv("submission.csv")

In [28]:
print(sub.shape)
print(sub.head())
print(sub.isnull().sum())

(41778, 2)
   Index    demand
0      0  0.052345
1      1  0.025225
2      2  0.003106
3      3  0.023834
4      4  0.049234
Index     0
demand    0
dtype: int64


In [29]:
pd.read_csv("submission.csv").head()

,Index,demand
0,0,0.052345
1,1,0.025225
2,2,0.003106
3,3,0.023834
4,4,0.049234


In [3]:
import pandas as pd

train = pd.read_csv("Dataset/train.csv")
test = pd.read_csv("Dataset/test.csv")

print(train.shape)
print(test.shape)

(77299, 11)
(41778, 10)


In [4]:
lookup = train.groupby(
    ['geohash','day','timestamp']
)['demand'].agg(['count','mean','std'])

print(lookup.head())
print("Unique groups:", len(lookup))

                       count      mean  std
geohash day timestamp                      
qp02yc  48  10:30          1  0.046790  NaN
            10:45          1  0.021158  NaN
            1:0            1  0.005397  NaN
            2:30           1  0.012944  NaN
            2:45           1  0.025961  NaN
Unique groups: 77299


In [5]:
print(
    train.groupby(
        ['geohash']
    )['demand'].agg(['mean','std']).head()
)

             mean       std
geohash                    
qp02yc   0.018498  0.012374
qp02yf   0.029433       NaN
qp02yy   0.002902  0.001435
qp02yz   0.036564  0.022065
qp02z1   0.040048  0.030124


In [6]:
print(
    train.groupby(
        ['timestamp']
    )['demand'].mean().sort_values(
        ascending=False
    ).head(20)
)

timestamp
13:30    0.119193
11:15    0.119174
14:0     0.117821
12:0     0.117790
11:30    0.117561
11:45    0.117129
13:45    0.117046
13:15    0.116641
11:0     0.115395
12:15    0.114676
10:45    0.114491
12:45    0.113965
13:0     0.112664
12:30    0.112614
9:15     0.112049
14:15    0.111349
10:30    0.111195
10:15    0.110334
10:0     0.110319
9:30     0.109178
Name: demand, dtype: float64


In [7]:
print(
    train.groupby(
        ['RoadType']
    )['demand'].mean()
)

RoadType
Highway        0.610756
Residential    0.057209
Street         0.273164
Name: demand, dtype: float64


In [8]:
print(
    train.groupby('timestamp')['demand']
    .mean()
    .sort_values(ascending=False)
    .head(20)
)

timestamp
13:30    0.119193
11:15    0.119174
14:0     0.117821
12:0     0.117790
11:30    0.117561
11:45    0.117129
13:45    0.117046
13:15    0.116641
11:0     0.115395
12:15    0.114676
10:45    0.114491
12:45    0.113965
13:0     0.112664
12:30    0.112614
9:15     0.112049
14:15    0.111349
10:30    0.111195
10:15    0.110334
10:0     0.110319
9:30     0.109178
Name: demand, dtype: float64


In [9]:
print(
    train.groupby('geohash')['demand']
    .mean()
    .describe()
)

count    1249.000000
mean        0.064671
std         0.099861
min         0.000495
25%         0.012661
50%         0.030807
75%         0.075227
max         0.960715
Name: demand, dtype: float64
